# Polynomial Regression

**Topic:** Supervised Learning — Regression

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import IntSlider, ToggleButtons, Output, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how polynomial regression builds a curve by adding powers of a feature and then running ordinary linear regression
- **Read** a train vs. validation curve and identify where overfitting begins
- **Explain** why the same degree overfits on a small dataset and not on a large one

> **Tip:** In **Building a Curve from Simple Shapes**, watch how each added degree contributes a new faint curve, and how those faint curves stack into the solid fit. Then in **Where Overfitting Starts**, set the degree slider high (10-12) and flip the **Training data** toggle between 20, 100, and 2000 points — the same high degree that falls apart at 20 points holds together at 2000.

---
## How we got here

Polynomial regression is linear regression applied to expanded features:

- **[supervised/02_multiple_linear_regression.ipynb](02_multiple_linear_regression.ipynb)** — the matrix formulation and normal equations you need; polynomial regression is multiple regression on engineered inputs
- **[ml_concepts/05_overfitting_and_underfitting.ipynb](../ml_concepts/05_overfitting_and_underfitting.ipynb)** — the core tension this notebook makes visual: low degree underfits, high degree overfits
- **[feature_engineering/05_polynomial_features.ipynb](../feature_engineering/05_polynomial_features.ipynb)** — how sklearn's PolynomialFeatures transformer builds the expanded feature matrix

---
## Why this matters for data science

Many real-world relationships are not straight lines. Sales growth accelerates before leveling off. Drug efficacy peaks at a dose and then declines. Housing prices grow faster at the high end than the low end. Polynomial regression handles these curves while staying within the linear model family — the transformed features are nonlinear, but the coefficients are still fit linearly.

It also introduces one of the most important concepts in all of machine learning: the degree hyperparameter directly controls model complexity, and choosing it requires cross-validation. Everything you learn about selecting degree here generalizes to every complexity-controlling hyperparameter you will tune later.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Polynomial regression sits in the **middle of the spectrum**, drifting right as degree increases. At degree 1 it is identical to multiple linear regression — fully interpretable. Degree 2 or 3 on a handful of features is still readable, one coefficient per power. Interpretability degrades as degree and feature count climb together — push either number up and the coefficients stop being something anyone reads individually.

The practical lesson is that model complexity is a dial, not a binary. You choose how far right you move by choosing degree, and cross-validation tells you when you have gone too far.

---
## How it learns

Imagine you are trying to fit a curved line through a scatter of points. A straight line leaves a clear curved pattern in the residuals — it is systematically too low in the middle and too high at the ends. You need more flexibility.

The trick is to create new features from the ones you already have. For a single feature $x$, you generate $x^2$, $x^3$, and so on. Then you hand this expanded feature matrix to ordinary linear regression. The algorithm still learns a weighted sum — but now the weights apply to powers of $x$, so the result can be any polynomial curve.

The catch is flexibility. Give the model enough powers of x and it can bend to touch nearly every training point, especially when there aren't many of them. The result looks perfect on the data it learned from and ridiculous everywhere else.

---
## The math behind it

Polynomial regression applies a **feature map** $\phi$ to the inputs before linear regression:

$$\phi(x) = [1,\; x,\; x^2,\; x^3,\; \ldots,\; x^d]$$

For a single feature, this gives a degree-$d$ polynomial. The model is still linear in the coefficients:

$$\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \cdots + \theta_d x^d = \boldsymbol{\theta}^\top \phi(x)$$

For $p$ features, PolynomialFeatures also adds **interaction terms** ($x_1 x_2$, $x_1^2 x_2$, etc.). The feature count grows fast: with 5 features, degree 3 turns those 5 columns into 55; degree 4 turns them into 125.

**Loss function** — same MSE as linear regression, minimized via the normal equations on the expanded feature matrix:

$$J(\boldsymbol{\theta}) = \frac{1}{n} \sum_{i=1}^{n} \left(\boldsymbol{\theta}^\top \phi(\mathbf{x}_i) - y_i\right)^2$$

**Optimization criterion:** minimize MSE. The degree $d$ is a hyperparameter you choose using cross-validation — the normal equations find the best coefficients for any fixed $d$, but cannot choose $d$ themselves.

---
## Try it yourself

Two widgets below: **Building a Curve from Simple Shapes** shows what each added polynomial degree contributes to the fitted curve, and **Where Overfitting Starts** shows what happens to training and validation performance as degree climbs and as the amount of training data changes.

### Building a Curve from Simple Shapes

In [2]:
np.random.seed(42)
_n_synth_w1 = 60
_x_synth_w1 = np.linspace(-1, 1, _n_synth_w1)
_true_w1 = 0.4 + 0.8 * _x_synth_w1 + 1.6 * _x_synth_w1 ** 3
_y_synth_w1 = _true_w1 + np.random.normal(0, 0.15, _n_synth_w1)

out_basis = Output()

degree_sl_w1 = IntSlider(value=3, min=1, max=8, step=1,
    description="Degree:", style={"description_width": "70px"},
    layout=widgets.Layout(width="380px"))

_BASIS_COLORS = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"],
                  PALETTE["muted"], "#AE3EC9", "#0CA678", "#F08C00", "#1864AB"]

_NUMBER_WORDS = {1: "One", 2: "Two", 3: "Three", 4: "Four", 5: "Five",
                 6: "Six", 7: "Seven", 8: "Eight"}
_SUPERSCRIPTS = {1: "", 2: "\u00b2", 3: "\u00b3", 4: "\u2074", 5: "\u2075",
                 6: "\u2076", 7: "\u2077", 8: "\u2078"}

def render_basis(change=None):
    d = degree_sl_w1.value
    pf = PolynomialFeatures(degree=d, include_bias=False)
    Xp = pf.fit_transform(_x_synth_w1.reshape(-1, 1))
    lr = LinearRegression().fit(Xp, _y_synth_w1)

    x_line = np.linspace(-1, 1, 200)
    fit_line = lr.intercept_ + sum(lr.coef_[k - 1] * x_line ** k for k in range(1, d + 1))

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "The pieces and the curve they add up to", "Learned weights"))

    fig.add_trace(go.Scatter(
        x=_x_synth_w1, y=_y_synth_w1, mode="markers",
        marker=dict(color=PALETTE["muted"], size=5, opacity=0.4),
        name="Data", showlegend=False,
    ), row=1, col=1)

    for k in range(1, d + 1):
        contribution = lr.coef_[k - 1] * x_line ** k
        fig.add_trace(go.Scatter(
            x=x_line, y=contribution, mode="lines",
            line=dict(color=_BASIS_COLORS[(k - 1) % len(_BASIS_COLORS)], width=1),
            opacity=0.45, name=f"\u03b8_{k}\u00b7x^{k}", showlegend=False,
        ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=x_line, y=fit_line, mode="lines",
        line=dict(color=PALETTE["primary"], width=3), name=f"Degree {d} fit",
        showlegend=False,
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=[f"x^{k}" for k in range(1, d + 1)], y=lr.coef_,
        marker_color=PALETTE["primary"], showlegend=False,
    ), row=1, col=2)

    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_xaxes(title_text="Term", row=1, col=2)
    fig.update_yaxes(title_text="Weight", row=1, col=2)

    fig.update_layout(
        title=dict(text=f"Degree {d} \u2014 {d} scaled term(s), {d} learned weight(s)",
                    font=dict(size=FONT["size_title"], family=FONT["family"])),
        height=440, paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
        font=dict(family=FONT["family"]),
    )

    terms = " + ".join(f"{lr.coef_[k-1]:.2f}\u00b7x{_SUPERSCRIPTS[k]}" for k in range(1, d + 1))
    number_word = _NUMBER_WORDS[d]
    plural = "s" if d > 1 else ""
    caption = (f"Degree {d}: this curve is {terms} plus an intercept. "
               f"{number_word} shape{plural}, {number_word.lower()} weight{plural}, one curve.")
    fig.add_annotation(
        text=caption, xref="paper", yref="paper", x=0.5, y=-0.22,
        showarrow=False, align="center",
        font=dict(size=13, family=FONT["family"], color=PALETTE["muted"]),
    )
    fig.update_layout(margin=dict(b=100))

    with out_basis:
        clear_output(wait=True)
        fig.show()

degree_sl_w1.observe(render_basis, names="value")
display(VBox([degree_sl_w1, out_basis]))
render_basis()

> Slide from 1 to 8 and watch the **x** bar specifically. It doesn't settle. That's because x, x², and x³ all carry nearly the same information over this range, so the model can shuffle credit between them almost freely. This is why you should never read a single coefficient off a high-degree polynomial.

### Where Overfitting Starts

In [3]:
_h_w2b = fetch_california_housing(as_frame=True)
_x_all_w2b = _h_w2b.data["MedInc"].values
_y_all_w2b = _h_w2b.target.values

out_boundary = Output()

degree_sl_w2 = IntSlider(value=2, min=1, max=12, step=1,
    description="Degree:", style={"description_width": "100px"},
    layout=widgets.Layout(width="380px"))

n_tb_w2 = ToggleButtons(
    options=[("20 points", 20), ("100 points", 100), ("2000 points", 2000)],
    value=100, description="Training data:",
    style={"description_width": "100px"},
)

_cache_w2 = {}

def _compute_for_n(n):
    rs = np.random.RandomState(42)
    idx = rs.choice(len(_x_all_w2b), n, replace=False)
    x_n, y_n = _x_all_w2b[idx], _y_all_w2b[idx]
    xtr, xva, ytr, yva = train_test_split(x_n, y_n, test_size=0.3, random_state=42)
    train_r2s, val_r2s, models = [], [], []
    for d in range(1, 13):
        pipe = Pipeline([
            ("poly", PolynomialFeatures(degree=d, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ])
        pipe.fit(xtr.reshape(-1, 1), ytr)
        train_r2s.append(r2_score(ytr, pipe.predict(xtr.reshape(-1, 1))))
        val_r2s.append(r2_score(yva, pipe.predict(xva.reshape(-1, 1))))
        models.append(pipe)
    return dict(x=x_n, y=y_n, xtr=xtr, xva=xva, ytr=ytr, yva=yva,
                train_r2s=train_r2s, val_r2s=val_r2s, models=models)

def _fit_tier(gap, val_r2):
    if val_r2 < 0:
        return "Severe overfitting", PALETTE["secondary"]
    elif gap > 0.1:
        return "Mild overfitting", "#F59F00"
    else:
        return "Good fit", PALETTE["accent"]

def render_boundary(change=None):
    n = n_tb_w2.value
    d = degree_sl_w2.value
    if n not in _cache_w2:
        _cache_w2[n] = _compute_for_n(n)
    data = _cache_w2[n]

    pipe = data["models"][d - 1]
    train_r2 = data["train_r2s"][d - 1]
    val_r2 = data["val_r2s"][d - 1]
    gap = train_r2 - val_r2

    x_line = np.linspace(data["x"].min(), data["x"].max(), 300)
    y_line = pipe.predict(x_line.reshape(-1, 1))

    tier_label, tier_color = _fit_tier(gap, val_r2)
    val_r2_str = f"{val_r2:.3f}" if abs(val_r2) < 100 else f"{val_r2:.2e}"

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"Degree {d} fit  |  n={n}", "Train vs. validation R\u00b2 by degree"))

    fig.add_trace(go.Scatter(
        x=data["xtr"], y=data["ytr"], mode="markers",
        marker=dict(color=PALETTE["muted"], size=5, opacity=0.4),
        name="Training", showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=data["xva"], y=data["yva"], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=5, opacity=0.5),
        name="Validation", showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line, mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name=f"Degree {d} fit", showlegend=False,
    ), row=1, col=1)

    fig.update_yaxes(range=[-1, 7], title_text="Median House Value", row=1, col=1)
    fig.update_xaxes(title_text="MedInc", row=1, col=1)

    if y_line.min() < -1 or y_line.max() > 7:
        fig.add_annotation(
            text="curve continues off-chart", xref="x domain", yref="y domain",
            x=0.5, y=0.95, row=1, col=1, showarrow=False,
            font=dict(size=11, color=PALETTE["secondary"]),
        )

    degrees = list(range(1, 13))
    fig.add_trace(go.Scatter(
        x=degrees, y=[max(v, -1.5) for v in data["train_r2s"]], mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2), marker=dict(size=6),
        name="Train R\u00b2",
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=degrees, y=[max(v, -1.5) for v in data["val_r2s"]], mode="lines+markers",
        line=dict(color=PALETTE["secondary"], width=2), marker=dict(size=6),
        name="Validation R\u00b2",
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[d, d], y=[-1.5, 1.05], mode="lines",
        line=dict(color=PALETTE["muted"], width=1.5, dash="dash"),
        name=f"Current degree ({d})", showlegend=False,
    ), row=1, col=2)

    fig.update_yaxes(range=[-1.5, 1.05], title_text="R\u00b2 (clipped)", row=1, col=2)
    fig.update_xaxes(title_text="Degree", tickmode="linear", dtick=1, row=1, col=2)

    fig.update_layout(
        title=dict(text=f"Degree {d}  |  Train R\u00b2={train_r2:.3f}  Val R\u00b2={val_r2_str}",
                    font=dict(size=FONT["size_title"], family=FONT["family"])),
        height=460, paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
        font=dict(family=FONT["family"]), showlegend=True,
    )
    fig.add_annotation(
        x=0.02, y=1.12, xref="x domain", yref="y domain", row=1, col=1,
        showarrow=False, text=f"<b>{tier_label}</b>",
        font=dict(size=12, color=tier_color), align="left",
    )

    n_label = f"{n} training points"
    if n == 20:
        note = " With this little data, one train/validation split is mostly noise."
    else:
        if 20 not in _cache_w2:
            _cache_w2[20] = _compute_for_n(20)
        ref_val = _cache_w2[20]["val_r2s"][d - 1]
        if val_r2 >= -0.2 and ref_val < -0.5:
            note = f" At 20 points this same degree collapses to validation R\u00b2 {ref_val:.2f} or worse."
        elif val_r2 < -0.2:
            note = " The gap between train and validation shows up at 20 points too, just earlier."
        else:
            note = " The two curves are still close together here."

    caption = (f"{n_label}, degree {d}: train R\u00b2 {train_r2:.3f}, validation R\u00b2 {val_r2_str}."
               + note)
    fig.add_annotation(
        text=caption, xref="paper", yref="paper", x=0.5, y=-0.24,
        showarrow=False, align="center",
        font=dict(size=13, family=FONT["family"], color=PALETTE["muted"]),
    )
    fig.update_layout(margin=dict(b=110))

    with out_boundary:
        clear_output(wait=True)
        fig.show()

degree_sl_w2.observe(render_boundary, names="value")
n_tb_w2.observe(render_boundary, names="value")
display(VBox([degree_sl_w2, n_tb_w2, out_boundary]))
render_boundary()

Notice the validation curve is bumpy, and at 20 points it jumps around a lot. One train/validation split is one roll of the dice. The cross-validation chart at the bottom of this notebook is how we stop guessing.

---
## What's happening?

In **Building a Curve from Simple Shapes**, the left panel shows what each degree actually contributes: every faint line is one power of x, scaled by its learned weight, and the solid curve on top is just those faint lines added together. Degree 1 is a single scaled line, degree 2 adds a scaled parabola, degree 3 adds a scaled inflection, and so on — the normal equations find the weight for each shape that makes the sum match the data as closely as possible.

In **Where Overfitting Starts**, training performance is non-decreasing as degree climbs, because the model always has at least as many tools as before to trace the training data — it can never do worse with more flexibility available. Validation performance tells a different story: it holds up reasonably well through the low-to-mid degrees, then falls apart at high degree, and how badly depends on how much data you have. Why does it break at the edges specifically? Because powers of x are not local. x¹² is tiny near the center of the range and enormous near the ends, so the far edges of the data have outsized pull on the shape of the whole curve. And out at those edges there are only a few points, so the curve has almost nothing holding it down. It swings. This is also why the model is hopeless at predicting beyond the range it was trained on: the curve out there was never anchored by anything.

The right degree is where validation error is minimized. That balance point varies by dataset and by how much data you have, which is why you always select it using cross-validation rather than guessing — and why the real-world example below uses actual 5-fold cross-validation instead of a single lucky (or unlucky) train/validation split.

---
## Key hyperparameters

**`degree`** (default `2` in PolynomialFeatures) — the maximum polynomial degree. This is the most important hyperparameter. Select it using cross-validation, not by minimizing training error.

**`interaction_only`** (default `False`) — if `True`, only interaction terms ($x_i x_j$) are generated, not powers ($x_i^2$). Useful when you believe features interact but their individual curvatures are not important.

**`include_bias`** (default `True`) — whether to include the constant feature (column of ones). PolynomialFeatures emits this column of ones by default, and LinearRegression fits its own intercept, so you get a redundant column; set `include_bias=False` unless you have turned `fit_intercept=False` on the model.

For the full list of hyperparameters, see the sklearn documentation:
[https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Captures nonlinear relationships without changing the algorithm | Overfits aggressively at high degrees |
| Still interpretable at degree 2 or 3 | Feature count grows exponentially with degree and number of features |
| Works within the linear regression framework — fast and exact solution | Coefficients become uninterpretable at high degrees |
| Cross-validation gives a principled way to choose degree | Sensitive to outliers (high-degree polynomials swing wildly near extreme points) |
| Pipeline-compatible: PolynomialFeatures + LinearRegression | Extrapolates very poorly outside the training data range |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| The scatter plot shows a clear curved relationship | You have many features (exponential feature growth makes it impractical) |
| You need an interpretable nonlinear model (degree 2 is still readable) | The relationship has many inflections (splines or tree-based models are better) |
| You want to stay in the linear model framework for speed and simplicity | You need to extrapolate beyond the training range |
| Degree 2 or 3 captures the curvature you see visually | You are already seeing overfitting at degree 2 (insufficient data for nonlinearity) |
| You are building interaction features deliberately | Your priority is automatic feature selection (use Lasso instead) |

In [4]:
from sklearn.model_selection import KFold

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
_x_all = housing.data["MedInc"].values
_y_all = housing.target.values
# Same n=100 subsample rationale as the widgets above: enough rows to be a real
# demo, few enough that a high-degree single-feature polynomial can actually overfit.
_idx = np.random.RandomState(42).choice(len(_x_all), 100, replace=False)
x, y = _x_all[_idx], _y_all[_idx]

# Held out ONCE, untouched until the very end — this is what makes the final
# number trustworthy instead of optimistic.
x_pool, x_test, y_pool, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

degrees = list(range(1, 11))
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_means, cv_mins, cv_maxs = [], [], []

for d in degrees:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("sc", StandardScaler()),
        ("lr", LinearRegression()),
    ])
    scores = cross_val_score(pipe, x_pool.reshape(-1, 1), y_pool, cv=kf, scoring="r2")
    cv_means.append(scores.mean())
    cv_mins.append(scores.min())
    cv_maxs.append(scores.max())

best_d = degrees[int(np.argmax(cv_means))]

# Refit on the FULL cv pool at the selected degree, evaluate ONCE on the held-out test set
final_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=best_d, include_bias=False)),
    ("sc", StandardScaler()),
    ("lr", LinearRegression()),
])
final_pipe.fit(x_pool.reshape(-1, 1), y_pool)
test_r2 = r2_score(y_test, final_pipe.predict(x_test.reshape(-1, 1)))

print(f"Best degree by mean 5-fold CV R²: {best_d}  (CV R² = {cv_means[best_d - 1]:.3f})")
print(f"Held-out test R² at degree {best_d}: {test_r2:.3f}")
print()
print("The test set was never touched during degree selection - this is the number")
print("that actually reflects how the model will perform on new data.")

_floor = -1.5
plot_means = [max(m, _floor) for m in cv_means]
plot_mins  = [max(m, _floor) for m in cv_mins]
plot_maxs  = [max(m, _floor) for m in cv_maxs]

fig = go.Figure(data=[
    go.Scatter(
        x=degrees + degrees[::-1],
        y=plot_maxs + plot_mins[::-1],
        fill="toself", fillcolor="rgba(150,150,150,0.25)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Fold min-max range",
    ),
    go.Scatter(
        x=degrees, y=plot_means, mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2.5), marker=dict(size=8),
        name="Mean 5-fold CV R²",
    ),
    go.Scatter(
        x=[best_d], y=[plot_means[best_d - 1]], mode="markers",
        marker=dict(color=PALETTE["accent"], size=14, symbol="star"),
        name=f"Selected degree = {best_d}",
    ),
], layout=base_layout(
    title="Polynomial Degree vs Mean 5-Fold CV R² (values below -1.5 clipped for readability)",
    xaxis_title="Polynomial Degree",
    yaxis_title="R² (clipped at -1.5)",
))
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1), yaxis=dict(range=[-1.6, 1.05]))
fig.show()

Best degree by mean 5-fold CV R²: 1  (CV R² = 0.242)
Held-out test R² at degree 1: 0.525

The test set was never touched during degree selection - this is the number
that actually reflects how the model will perform on new data.


---
## Real-world example: choosing degree with cross-validation, not guesswork

**Where Overfitting Starts** showed how noisy a single train/validation split can be, especially with little data. The chart above fixes that: using MedInc alone on a 100-row sample, it runs genuine 5-fold cross-validation for degrees 1 through 10, then refits the selected degree on the full cross-validation pool and checks it exactly once against a held-out test set that was carved off before any of the degree selection happened.

- **Notice:** mean CV R² is fairly flat through degrees 1-5, with the shaded fold range showing real fold-to-fold variability even at low degree — a reminder that a single train/validation split (as opposed to 5 folds) can easily land on an unrepresentative number
- **Notice:** from around degree 6 onward, mean CV R² drops sharply and then goes deeply negative — the curve is swinging wildly out at the edges of the data, the same effect from the widget above, now compounding across every fold
- **Notice:** the selected degree (starred) is chosen purely from the cross-validation curve, before the held-out test set is touched at all

> **Discussion question:** the code above only checks the held-out test set once, after degree selection is already finished. What would happen to the trustworthiness of the final test R² if you instead tried several degrees directly against the test set and reported whichever looked best?

### Degree selection strategy

| Degree | Typical behavior | When to use |
|---|---|---|
| 1 | Straight line (linear) | When scatter plot is linear |
| 2 | Parabola (one bend) | Accelerating or decelerating trends |
| 3 | One inflection point | S-curves, growth-plateau patterns |
| 4+ | Multiple bends | Rarely worth it on its own. If you want high degree, pair it with regularization (next notebook) |

> **Polynomial regression adds curves to the linear model by engineering powers of the input features — powerful and interpretable at low degree, but it overfits quickly at high degree and always requires cross-validation to choose the right one.**

---
*Next up: 04 — Ridge and Lasso Regression, which add a penalty to prevent overfitting without sacrificing the linear framework*